In [1]:
import os
os.makedirs(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures", exist_ok=True)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

data = {
    ("RT-DETR-L", "Summer"): {"reported": 0.532, "mean": 0.532, "lo": 0.441, "hi": 0.629},
    ("RT-DETR-L", "Fall"):   {"reported": 0.205, "mean": 0.208, "lo": 0.158, "hi": 0.261},
    ("YOLOv8m",   "Summer"): {"reported": 0.520, "mean": 0.531, "lo": 0.459, "hi": 0.610},
    ("YOLOv8m",   "Fall"):   {"reported": 0.229, "mean": 0.232, "lo": 0.176, "hi": 0.298},
}

COLORS = {
    ("RT-DETR-L", "Summer"): "#1F6FBF",
    ("RT-DETR-L", "Fall"):   "#8B3A8B",
    ("YOLOv8m",   "Summer"): "#E07B00",
    ("YOLOv8m",   "Fall"):   "#C0392B",
}
MARKERS = {"RT-DETR-L": "o", "YOLOv8m": "s"}

def axis_limits(season):
    all_lo = min(data[(m, season)]["lo"] for m in ["RT-DETR-L", "YOLOv8m"])
    all_hi = max(data[(m, season)]["hi"] for m in ["RT-DETR-L", "YOLOv8m"])
    buf = (all_hi - all_lo) * 0.35
    return round(all_lo - buf, 2), round(all_hi + buf, 2)

SEASON_TITLE_COLORS = {"Summer": "#2E6DA4", "Fall": "#7B2D8B"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='white')

for ax, season in [(ax1, "Summer"), (ax2, "Fall")]:
    ax.set_facecolor('white')
    y_positions = {"RT-DETR-L": 1.0, "YOLOv8m": 0.6}

    for model, y in y_positions.items():
        d        = data[(model, season)]
        reported = d["reported"]
        mean_val = d["mean"]
        lo, hi   = d["lo"], d["hi"]
        color    = COLORS[(model, season)]
        marker   = MARKERS[model]

        # CI line
        ax.plot([lo, hi], [y, y], color=color, linewidth=2.5, zorder=2)

        # end caps
        for x_cap in [lo, hi]:
            ax.plot([x_cap, x_cap], [y-0.04, y+0.04],
                    color=color, linewidth=2.5, zorder=2)

        # bootstrap mean tick
        ax.plot([mean_val, mean_val], [y-0.07, y+0.07],
                color=color, linewidth=2.0, linestyle='--',
                zorder=4, alpha=0.8)
        ax.text(mean_val, y - 0.13,
                f"μ={mean_val:.3f}",
                ha='center', va='top',
                fontsize=11, color=color, alpha=0.85, style='italic')

        # reported mAP50 marker
        ax.scatter([reported], [y],
                   color=color, marker=marker,
                   s=130, zorder=5,
                   edgecolors='white', linewidths=1.5,
                   label=f"{model}")

        # reported value label above
        ax.text(reported, y + 0.09,
                f"{reported:.3f}",
                ha='center', va='bottom',
                fontsize=13, color=color, fontweight='bold')

        # CI bound labels
        ax.text(lo - 0.004, y, f"{lo:.3f}",
                ha='right', va='center', fontsize=11, color=color, alpha=0.8)
        ax.text(hi + 0.004, y, f"{hi:.3f}",
                ha='left',  va='center', fontsize=11, color=color, alpha=0.8)

    # CI overlap shading
    lo_max = max(data[("RT-DETR-L", season)]["lo"],
                 data[("YOLOv8m",   season)]["lo"])
    hi_min = min(data[("RT-DETR-L", season)]["hi"],
                 data[("YOLOv8m",   season)]["hi"])
    if lo_max < hi_min:
        ax.axvspan(lo_max, hi_min, alpha=0.08, color='grey',
                   label=f"CI overlap [{lo_max:.3f}, {hi_min:.3f}]")

    # marker explanation note
    ax.text(0.02, 0.04,
            "● / ■ = reported mAP50   ╎ = bootstrap mean (μ)",
            transform=ax.transAxes,
            fontsize=11, color='#666666', style='italic')

    ax.set_title(season, fontsize=16, fontweight='bold',
                 color=SEASON_TITLE_COLORS[season], pad=10)
    ax.set_xlabel("mAP50", fontsize=15, color='#333333')

    xlo, xhi = axis_limits(season)
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(0.25, 1.45)
    ax.set_yticks([])
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.grid(axis='x', alpha=0.3, color='#999999')
    ax.tick_params(colors='#333333', labelsize=13)
    ax.legend(fontsize=12, loc='upper right',
              framealpha=0.95, edgecolor='#cccccc',
              handlelength=1.5)

plt.tight_layout()
out = (rf"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures"
       rf"\forest_plot_bootstrap_ci_{timestamp}.png")
os.makedirs(os.path.dirname(out), exist_ok=True)
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
print(f"Saved: {out}")
plt.close()

Results saved to pkl
Computing RT-DETR v7 Summer...
  Best F1=0.563 at conf=0.482
Computing RT-DETR v7 Fall...
  Best F1=0.315 at conf=0.422
Computing YOLOv8m v6 Summer...
  Best F1=0.562 at conf=0.161
Computing YOLOv8m v6 Fall...
  Best F1=0.319 at conf=0.086
Saved PR curves
Saved F1 curves

Done. Check results/figures/ folder.


In [ ]:
# compute_pr_f1_curves.py
####just figures and using already computed numbers from above
# Purpose: Recompute PR and F1 curves from saved prediction files
# for RT-DETR v7 and YOLOv8m v6 on summer and fall test sets
# Produces two publication-ready figures:
#   1. PR curves (all 4 combinations overlaid)
#   2. F1 curves (all 4 combinations overlaid)

import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
import pickle
# save computed results
with open(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\pr_f1_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("Results saved to pkl")
plt.rcParams.update({
    'font.size':        14,
    'axes.titlesize':   15,
    'axes.labelsize':   14,
    'xtick.labelsize':  13,
    'ytick.labelsize':  13,
    'legend.fontsize':  12,
    'figure.titlesize': 18,
})

# ── paths ──────────────────────────────────────────────────────────────────────
PRED_DIRS = {
    ("RT-DETR-L", "Summer"): r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation\test_summer\labels",
    ("RT-DETR-L", "Fall"):   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation\test_fall\labels",
    ("YOLOv8m", "Summer"): r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation_yolo_v6\test_summer\labels",
    ("YOLOv8m", "Fall"):   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation_yolo_v6\test_fall\labels",
}

GT_DIRS = {
    "Summer": r"F:\Masterthesis\dataset\cv_folds_2cls\test\labels",
    "Fall":   r"F:\Masterthesis\dataset\cv_folds_fall_2cls\test\labels",
}

IOU_THRESH = 0.5
CLASSES    = [0, 1]  # 0=deciduous, 1=shrub
N_THRESH   = 200     # number of confidence thresholds to sweep

# ── helpers ────────────────────────────────────────────────────────────────────
def compute_iou(pred, gt):
    def corners(b):
        return b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    px1,py1,px2,py2 = corners(pred)
    gx1,gy1,gx2,gy2 = corners(gt)
    ix1,iy1 = max(px1,gx1), max(py1,gy1)
    ix2,iy2 = min(px2,gx2), min(py2,gy2)
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    union = (px2-px1)*(py2-py1)+(gx2-gx1)*(gy2-gy1)-inter
    return inter/union if union>0 else 0.0

def load_data(pred_dir, gt_dir):
    """Load all predictions and ground truths, matched by filename."""
    gt_files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".txt")])
    all_preds = []  # list of (conf, cls, tile_idx)
    all_gts   = defaultdict(list)  # tile_idx → list of (cls, cx, cy, w, h)
    n_gt = {c: 0 for c in CLASSES}

    for tile_idx, fname in enumerate(gt_files):
        # load GT
        with open(os.path.join(gt_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls = int(parts[0])
                if cls in CLASSES:
                    all_gts[tile_idx].append(
                        (cls, float(parts[1]), float(parts[2]),
                         float(parts[3]), float(parts[4])))
                    n_gt[cls] += 1

        # load predictions
        pred_path = os.path.join(pred_dir, fname)
        if not os.path.exists(pred_path): continue
        with open(pred_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6: continue
                cls = int(parts[0])
                if cls in CLASSES:
                    all_preds.append((
                        float(parts[5]),  # confidence
                        cls,
                        tile_idx,
                        float(parts[1]), float(parts[2]),
                        float(parts[3]), float(parts[4])
                    ))

    # sort by confidence descending
    all_preds.sort(key=lambda x: x[0], reverse=True)
    return all_preds, all_gts, n_gt, len(gt_files)

def compute_pr_curve(all_preds, all_gts, n_gt, n_tiles, conf_thresholds):
    """Compute precision and recall at each confidence threshold."""
    precisions, recalls = [], []

    for thresh in conf_thresholds:
        filtered = [p for p in all_preds if p[0] >= thresh]

        # match predictions to GT
        matched = defaultdict(lambda: defaultdict(list))
        for tile_idx in range(n_tiles):
            for cls in CLASSES:
                matched[tile_idx][cls] = [False] * sum(
                    1 for g in all_gts[tile_idx] if g[0] == cls)

        tp = fp = 0
        for pred in filtered:
            conf, cls, tile_idx, cx, cy, w, h = pred
            gt_boxes = [(g[1],g[2],g[3],g[4])
                        for g in all_gts[tile_idx] if g[0] == cls]
            gt_matched = matched[tile_idx][cls]

            best_iou, best_j = 0.0, -1
            for j, gt in enumerate(gt_boxes):
                iou = compute_iou([cx,cy,w,h], list(gt))
                if iou > best_iou:
                    best_iou, best_j = iou, j

            if best_iou >= IOU_THRESH and best_j >= 0 and not gt_matched[best_j]:
                tp += 1
                gt_matched[best_j] = True
            else:
                fp += 1

        total_gt = sum(n_gt.values())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec  = tp / total_gt  if total_gt > 0 else 0.0
        precisions.append(prec)
        recalls.append(rec)

    return np.array(precisions), np.array(recalls)

def compute_f1(precisions, recalls):
    return 2 * precisions * recalls / (precisions + recalls + 1e-8)

# ── compute curves ─────────────────────────────────────────────────────────────
import pickle
with open(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\pr_f1_results.pkl", "rb") as f:
    results = pickle.load(f)
print("Results loaded from pkl — skipping recomputation")

# ── plot settings ──────────────────────────────────────────────────────────────
COLORS = {
    ("RT-DETR-L", "Summer"): "#1F6FBF",
    ("RT-DETR-L", "Fall"):   "#8B3A8B",
    ("YOLOv8m", "Summer"): "#E07B00",
    ("YOLOv8m", "Fall"):   "#C0392B",
}
STYLES = {
    ("RT-DETR-L", "Summer"): "-",
    ("RT-DETR-L", "Fall"):   "--",
    ("YOLOv8m", "Summer"): "-",
    ("YOLOv8m", "Fall"):   "--",
}
MARKERS = {
    ("RT-DETR-L", "Summer"): "o",
    ("RT-DETR-L", "Fall"):   "o",
    ("YOLOv8m", "Summer"): "s",
    ("YOLOv8m", "Fall"):   "s",
}

# ── Figure 1: PR curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')

for season, ax, title in [("Summer", ax1, "Summer"), ("Fall", ax2, "Fall")]:
    ax.set_facecolor('white')
    for model in ["RT-DETR v7", "YOLOv8m v6"]:
        key = (model, season)
        r = results[key]
        # sort by recall for clean curve
        sort_idx = np.argsort(r["recall"])
        rec_sorted  = r["recall"][sort_idx]
        prec_sorted = r["precision"][sort_idx]

        # compute AUC
        auc = np.trapezoid(prec_sorted, rec_sorted)

        ax.plot(rec_sorted, prec_sorted,
                color=COLORS[key],
                linestyle=STYLES[key],
                linewidth=2.0,
                label=f"{model} (mAP50={auc:.3f})")

        # mark best F1 point
        best_idx    = r["f1"].argmax()
        best_recall = r["recall"][best_idx]
        best_prec   = r["precision"][best_idx]
        ax.scatter([best_recall], [best_prec],
           color=COLORS[key], marker=MARKERS[key],
           s=80, zorder=5,
           label=f"Best F1 ({model})")

    season_color = '#2E6DA4' if season == 'Summer' else '#8B4513'
    ax.set_title(season, fontsize=13, fontweight='bold',
                 color=season_color, pad=8)
    ax.set_xlabel("Recall", fontsize=12, color='#333333')
    ax.set_ylabel("Precision", fontsize=12, color='#333333')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10, loc='upper right',
              framealpha=0.9, edgecolor='#cccccc')
    ax.grid(True, alpha=0.3, color='#999999')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(colors='#333333')

plt.tight_layout()
plt.savefig(rf"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\pr_curves_comparison_{timestamp}.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("Saved PR curves")

# ── Figure 2: F1 curves ────────────────────────────────────────────────────────
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')


for season, ax, title in [("Summer", ax3, "Summer"), ("Fall", ax4, "Fall")]:
    ax.set_facecolor('white')
    for model in ["RT-DETR v7", "YOLOv8m v6"]:
        key = (model, season)
        r = results[key]
        best_f1  = r["best_f1"]
        best_thr = r["best_thresh"]

        ax.plot(r["thresholds"], r["f1"],
                color=COLORS[key],
                linestyle=STYLES[key],
                linewidth=2.0,
                label=f"{model} (F1={best_f1:.3f} @ {best_thr:.2f})")

        # mark peak
        ax.scatter([best_thr], [best_f1],
           color=COLORS[key], marker=MARKERS[key],
           s=80, zorder=5,
           label=f"Best F1 ({model})")

        # vertical line at conf=0.25
    ax.axvline(0.25, color='#555555', linestyle=':', linewidth=1.5,
               label='conf=0.25 (operational)')

    season_color = '#2E6DA4' if season == 'Summer' else '#8B4513'
    ax.set_title(title, fontsize=13, fontweight='bold',
                 color=season_color, pad=8)
    ax.set_xlabel("Confidence threshold", fontsize=12, color='#333333')
    ax.set_ylabel("F1 score", fontsize=12, color='#333333')
    ax.set_xlim(0, 1.0)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=10, loc='upper right',
              framealpha=0.9, edgecolor='#cccccc')
    ax.grid(True, alpha=0.3, color='#999999')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(colors='#333333')

plt.tight_layout()
plt.savefig(rf"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\f1_curves_comparison_{timestamp}.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("Saved F1 curves")
print("\nDone. Check results/figures/ folder.")

Results saved to pkl
Results loaded from pkl — skipping recomputation


KeyError: ('RT-DETR v7', 'Summer')

In [10]:
# compute_pr_f1_curves.py
# Purpose: Recompute PR and F1 curves from prediction files
# for RT-DETR-L and YOLOv8m on summer and fall test sets

import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime
import pickle

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

plt.rcParams.update({
    'font.size':        14,
    'axes.titlesize':   15,
    'axes.labelsize':   14,
    'xtick.labelsize':  13,
    'ytick.labelsize':  13,
    'legend.fontsize':  12,
    'figure.titlesize': 18,
})

# ── paths ──────────────────────────────────────────────────────────────────────
PRED_DIRS = {
    ("RT-DETR-L", "Summer"): r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation\test_summer\labels",
    ("RT-DETR-L", "Fall"):   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation\test_fall\labels",
    ("YOLOv8m",   "Summer"): r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation_yolo_v6\test_summer\labels",
    ("YOLOv8m",   "Fall"):   r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation_yolo_v6\test_fall\labels",
}

GT_DIRS = {
    "Summer": r"F:\Masterthesis\dataset\cv_folds_2cls\test\labels",
    "Fall":   r"F:\Masterthesis\dataset\cv_folds_fall_2cls\test\labels",
}

IOU_THRESH = 0.5
CLASSES    = [0, 1]  # 0=deciduous, 1=shrub
N_THRESH   = 200

# ── helpers ────────────────────────────────────────────────────────────────────
def compute_iou(pred, gt):
    def corners(b):
        return b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    px1,py1,px2,py2 = corners(pred)
    gx1,gy1,gx2,gy2 = corners(gt)
    ix1,iy1 = max(px1,gx1), max(py1,gy1)
    ix2,iy2 = min(px2,gx2), min(py2,gy2)
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    union = (px2-px1)*(py2-py1)+(gx2-gx1)*(gy2-gy1)-inter
    return inter/union if union>0 else 0.0

def load_data(pred_dir, gt_dir):
    gt_files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".txt")])
    all_preds = []
    all_gts   = defaultdict(list)
    n_gt = {c: 0 for c in CLASSES}

    for tile_idx, fname in enumerate(gt_files):
        with open(os.path.join(gt_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls = int(parts[0])
                if cls in CLASSES:
                    all_gts[tile_idx].append(
                        (cls, float(parts[1]), float(parts[2]),
                         float(parts[3]), float(parts[4])))
                    n_gt[cls] += 1

        pred_path = os.path.join(pred_dir, fname)
        if not os.path.exists(pred_path): continue
        with open(pred_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6: continue
                cls = int(parts[0])
                if cls in CLASSES:
                    all_preds.append((
                        float(parts[5]),
                        cls,
                        tile_idx,
                        float(parts[1]), float(parts[2]),
                        float(parts[3]), float(parts[4])
                    ))

    all_preds.sort(key=lambda x: x[0], reverse=True)
    return all_preds, all_gts, n_gt, len(gt_files)

def compute_pr_curve(all_preds, all_gts, n_gt, n_tiles, conf_thresholds):
    precisions, recalls = [], []

    for thresh in conf_thresholds:
        filtered = [p for p in all_preds if p[0] >= thresh]

        matched = defaultdict(lambda: defaultdict(list))
        for tile_idx in range(n_tiles):
            for cls in CLASSES:
                matched[tile_idx][cls] = [False] * sum(
                    1 for g in all_gts[tile_idx] if g[0] == cls)

        tp = fp = 0
        for pred in filtered:
            conf, cls, tile_idx, cx, cy, w, h = pred
            gt_boxes = [(g[1],g[2],g[3],g[4])
                        for g in all_gts[tile_idx] if g[0] == cls]
            gt_matched = matched[tile_idx][cls]

            best_iou, best_j = 0.0, -1
            for j, gt in enumerate(gt_boxes):
                iou = compute_iou([cx,cy,w,h], list(gt))
                if iou > best_iou:
                    best_iou, best_j = iou, j

            if best_iou >= IOU_THRESH and best_j >= 0 and not gt_matched[best_j]:
                tp += 1
                gt_matched[best_j] = True
            else:
                fp += 1

        total_gt = sum(n_gt.values())
        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec  = tp / total_gt  if total_gt > 0 else 0.0
        precisions.append(prec)
        recalls.append(rec)

    return np.array(precisions), np.array(recalls)

def compute_f1(precisions, recalls):
    return 2 * precisions * recalls / (precisions + recalls + 1e-8)

# ── compute curves ─────────────────────────────────────────────────────────────
results = {}
conf_thresholds = np.linspace(0, 1, N_THRESH)

for (model, season), pred_dir in PRED_DIRS.items():
    print(f"Computing {model} / {season} ...")
    gt_dir = GT_DIRS[season]
    all_preds, all_gts, n_gt, n_tiles = load_data(pred_dir, gt_dir)
    precisions, recalls = compute_pr_curve(all_preds, all_gts, n_gt, n_tiles, conf_thresholds)
    f1 = compute_f1(precisions, recalls)
    best_idx = f1.argmax()
    results[(model, season)] = {
        "precision":   precisions,
        "recall":      recalls,
        "f1":          f1,
        "thresholds":  conf_thresholds,
        "best_f1":     f1[best_idx],
        "best_thresh": conf_thresholds[best_idx],
    }

with open(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\pr_f1_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("Results saved to pkl")

# ── plot settings ──────────────────────────────────────────────────────────────
COLORS = {
    ("RT-DETR-L", "Summer"): "#1F6FBF",
    ("RT-DETR-L", "Fall"):   "#8B3A8B",
    ("YOLOv8m",   "Summer"): "#E07B00",
    ("YOLOv8m",   "Fall"):   "#C0392B",
}
STYLES = {
    ("RT-DETR-L", "Summer"): "-",
    ("RT-DETR-L", "Fall"):   "--",
    ("YOLOv8m",   "Summer"): "-",
    ("YOLOv8m",   "Fall"):   "--",
}
MARKERS = {
    ("RT-DETR-L", "Summer"): "o",
    ("RT-DETR-L", "Fall"):   "o",
    ("YOLOv8m",   "Summer"): "s",
    ("YOLOv8m",   "Fall"):   "s",
}

# ── Figure 1: PR curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')

for season, ax in [("Summer", ax1), ("Fall", ax2)]:
    ax.set_facecolor('white')
    for model in ["RT-DETR-L", "YOLOv8m"]:
        key = (model, season)
        r = results[key]
        sort_idx    = np.argsort(r["recall"])
        rec_sorted  = r["recall"][sort_idx]
        prec_sorted = r["precision"][sort_idx]
        auc = np.trapezoid(prec_sorted, rec_sorted)

        ax.plot(rec_sorted, prec_sorted,
                color=COLORS[key],
                linestyle=STYLES[key],
                linewidth=2.0,
                label=f"{model} (mAP50={auc:.3f})")

        best_idx    = r["f1"].argmax()
        best_recall = r["recall"][best_idx]
        best_prec   = r["precision"][best_idx]
        ax.scatter([best_recall], [best_prec],
                   color=COLORS[key], marker=MARKERS[key],
                   s=80, zorder=5,
                   label=f"Best F1 ({model})")

    season_color = '#2E6DA4' if season == 'Summer' else '#8B4513'
    ax.set_title(season, fontsize=14, fontweight='bold',
                 color=season_color, pad=8)
    ax.set_xlabel("Recall", fontsize=12, color='#333333')
    ax.set_ylabel("Precision", fontsize=12, color='#333333')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=12, loc='upper right',
              framealpha=0.9, edgecolor='#cccccc')
    ax.grid(True, alpha=0.3, color='#999999')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(colors='#333333')

plt.tight_layout()
plt.savefig(rf"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\pr_curves_comparison_{timestamp}.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("Saved PR curves")

# ── Figure 2: F1 curves ────────────────────────────────────────────────────────
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')

for season, ax in [("Summer", ax3), ("Fall", ax4)]:
    ax.set_facecolor('white')
    for model in ["RT-DETR-L", "YOLOv8m"]:
        key = (model, season)
        r = results[key]
        best_f1  = r["best_f1"]
        best_thr = r["best_thresh"]

        ax.plot(r["thresholds"], r["f1"],
                color=COLORS[key],
                linestyle=STYLES[key],
                linewidth=2.0,
                label=f"{model}")

        ax.scatter([best_thr], [best_f1],
                   color=COLORS[key], marker=MARKERS[key],
                   s=80, zorder=5,
                   label=f"Best F1 ({model}, F1={best_f1:.3f})")

    ax.axvline(0.25, color='#555555', linestyle=':', linewidth=1.5,
               label='conf=0.25 (operational)')

    season_color = '#2E6DA4' if season == 'Summer' else '#8B4513'
    ax.set_title(season, fontsize=14, fontweight='bold',
                 color=season_color, pad=8)
    ax.set_xlabel("Confidence threshold", fontsize=12, color='#333333')
    ax.set_ylabel("F1 score", fontsize=12, color='#333333')
    ax.set_xlim(0, 1.0)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=12, loc='upper right',
              framealpha=0.9, edgecolor='#cccccc')
    ax.grid(True, alpha=0.3, color='#999999')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(colors='#333333')

plt.tight_layout()
plt.savefig(rf"C:\Master Physische Geographie\Canada\Masterarbeit\results\figures\f1_curves_comparison_{timestamp}.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("Saved F1 curves")
print("\nDone. Check results/figures/ folder.")

Computing RT-DETR-L / Summer ...
Computing RT-DETR-L / Fall ...
Computing YOLOv8m / Summer ...
Computing YOLOv8m / Fall ...
Results saved to pkl
Saved PR curves
Saved F1 curves

Done. Check results/figures/ folder.
